# Project Name

## Motivation

Identifying students who may experience elevated depressive symptoms later in the term could help universities offer support earlier. Start-of-term depression, loneliness, and flourishing surveys may provide useful information about students’ subsequent mental health.

## Objective

Evaluate whether start-of-term depression, loneliness, and flourishing scores can predict end-of-term PHQ-9 depression scores. Compare a model using baseline depression alone with a model that also includes loneliness and flourishing.

## Business Value

The study could provide preliminary evidence about whether adding loneliness and flourishing questionnaires improves universities’ ability to identify future support needs beyond a baseline depression survey. This could inform the design of screening and outreach programs, subject to larger-scale validation.

## Research question

Do start-of-term loneliness and flourishing scores improve prediction of end-of-term depressive symptom severity beyond start-of-term depression scores?

In [1]:
import sys

assert sys.version_info >= (3, 7)

In [2]:
from packaging import version
import sklearn

assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

### Data Acquisition

We are using the StudentLife dataset, accessed through Kaggle. The data originated from a Dartmouth College study conducted during the spring term of 2013, following 48 students over 10 weeks. The study examined students’ mental health, academic performance, and behavioral patterns using smartphone sensing and surveys.

Students completed mental health questionnaires at the beginning and end of the study. This longitudinal structure allows us to investigate whether information collected at the start of term can predict depressive symptoms at the end. Our project focuses on three survey files from the same study.

In [11]:
%pip install "kagglehub[pandas-datasets]"

import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

def load_studentlife_data(file_name):
    return kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "dartweichen/student-life",
        f"dataset/survey/{file_name}"
    )

Note: you may need to restart the kernel to use updated packages.


#### **Selected Survey Files**

1. PHQ-9.csv — Depressive Symptoms

The **Patient Health Questionnaire-9 (PHQ-9)** measures depressive symptom severity through nine items. Using standard scoring, the total ranges from **0 to 27**, with higher scores indicating greater symptom severity.

**How we will use it:**

- **Pre-survey PHQ-9 score:** Predictor representing baseline depressive symptoms.
- **Post-survey PHQ-9 score:** Target representing end-of-term depressive symptoms.

**Why we selected it:**  
Baseline depression provides a benchmark for predicting later symptoms. Including it allows us to assess whether loneliness and flourishing improve prediction beyond what is already explained by initial depression scores.

In [12]:
phq9_df = load_studentlife_data("PHQ-9.csv")
phq9_df.head()

,uid,type,Little interest or pleasure in doing things,"Feeling down, depressed, hopeless.","Trouble falling or staying asleep, or sleeping too much.",Feeling tired or having little energy,Poor appetite or overeating,Feeling bad about yourself or that you are a failure or have let yourself or your family down,"Trouble concentrating on things, such as reading the newspaper or watching television",Moving or speaking so slowly that other people could have noticed. Or the opposite being so figety or restless that you have been moving around a lot more than usual,"Thoughts that you would be better off dead, or of hurting yourself",Response
0,u00,pre,Not at all,Several days,Not at all,Several days,Not at all,Not at all,Not at all,Not at all,Not at all,Not difficult at all
1,u01,pre,Several days,Several days,Several days,Several days,Not at all,Several days,Not at all,Not at all,Not at all,Very difficult
2,u02,pre,More than half the days,Several days,More than half the days,More than half the days,More than half the days,Several days,Several days,More than half the days,Not at all,Somewhat difficult
3,u03,pre,Not at all,Several days,Not at all,Not at all,Not at all,Not at all,Not at all,Several days,Not at all,Somewhat difficult
4,u04,pre,Several days,Several days,Not at all,Several days,Several days,Several days,Several days,Not at all,Not at all,Somewhat difficult


2. LonelinessScale.csv — Perceived Loneliness

The **UCLA Loneliness Scale** measures perceived loneliness and social isolation. Higher scores indicate greater loneliness after applying the published scoring method.

**How we will use it:**

- Use only the **pre-survey loneliness score** as a predictor.

**Why we selected it:**  
This measure allows us to test whether students’ perceived social disconnection adds predictive information about end-of-term depressive symptoms beyond baseline depression.

In [13]:
loneliness_df = load_studentlife_data("LonelinessScale.csv")
loneliness_df.head()

,uid,type,1. I feel in tune with the people around me,2. I lack companionship,3. There is no one I can turn to,4. I do not feel alone,5. I feel part of a group of friends,6. I have a lot in common with the people around me,7. I am no longer close to anyone,8. My interests and ideas are not shared by those around me,...,11. I feel left out,12. My social relationships are superficial,13. No one really knows me well,14. I feel isolated from others,15. I can find companionship when I want it,16. There are people who really understand me,17. I am unhappy being so withdrawn,18. People are around me but not with me,19. There are people I can talk to,20. There are people I can turn to
0,u00,pre,Sometimes,Rarely,Never,Never,Often,Sometimes,Never,Never,...,Rarely,Rarely,Never,Never,Often,Rarely,Often,Sometimes,Often,Often
1,u01,pre,Sometimes,Rarely,Rarely,Sometimes,Often,Often,Never,Never,...,Rarely,Rarely,Never,Rarely,Often,Often,Sometimes,Never,Often,Often
2,u02,pre,Sometimes,Rarely,Rarely,Often,Often,Often,Never,Often,...,Rarely,Never,Never,Never,Often,Often,Never,Never,Often,Often
3,u03,pre,Sometimes,Rarely,Never,Often,Rarely,Rarely,Sometimes,Sometimes,...,Rarely,Rarely,Sometimes,Sometimes,Sometimes,Often,Sometimes,Sometimes,Sometimes,Sometimes
4,u04,pre,Sometimes,Sometimes,Sometimes,Sometimes,Rarely,Sometimes,Never,Rarely,...,Sometimes,Sometimes,Sometimes,Sometimes,Rarely,Rarely,Sometimes,Sometimes,Rarely,Rarely


3. FlourishingScale.csv — Psychological Well-Being

The **Flourishing Scale** measures psychological well-being through eight items covering areas such as relationships, purpose, self-esteem, and optimism. Standard total scores range from **8 to 56**, with higher scores indicating greater flourishing.

**How we will use it:**

- Use only the **pre-survey flourishing score** as a predictor.

**Why we selected it:**  
Flourishing adds a positive well-being perspective, complementing the symptom-focused depression and loneliness measures. We will assess whether it improves prediction of end-of-term depressive symptoms.


In [14]:
flourishing_df = load_studentlife_data("FlourishingScale.csv")
flourishing_df.head()

,uid,type,I lead a purposeful and meaningful life,My social relationships are supportive and rewarding,I am engaged and interested in my daily activities,I actively contribute to the happiness and well-being of others,I am competent and capable in the activities that are important to me,I am a good person and live a good life,I am optimistic about my future,People respect me
0,u00,pre,7,6,NaN,6.0,6.0,7.0,7,4
1,u01,pre,4,6,6.0,6.0,7.0,6.0,4,6
2,u02,pre,6,7,5.0,6.0,5.0,6.0,4,7
3,u03,pre,6,2,2.0,4.0,5.0,6.0,5,4
4,u04,pre,3,5,4.0,3.0,2.0,2.0,3,5


### Data Exploration and Cleaning

1. PHQ-9.csv — Depressive Symptoms

In [17]:
print(phq9_df.shape)
phq9_df.info()

(84, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 12 columns):
 #   Column                                                                                                                                                                 Non-Null Count  Dtype 
---  ------                                                                                                                                                                 --------------  ----- 
 0   uid                                                                                                                                                                    84 non-null     object
 1   type                                                                                                                                                                   84 non-null     object
 2   Little interest or pleasure in doing things                                                            

In [18]:
# Missing values by column
missing_summary = pd.DataFrame({
    "MissingCount": phq9_df.isna().sum(),
    "MissingPercent": phq9_df.isna().mean() * 100
})

display(missing_summary)

# Exact duplicate rows
print("Duplicate rows:", phq9_df.duplicated().sum())

,MissingCount,MissingPercent
uid,0,0.000000
type,0,0.000000
Little interest or pleasure in doing things,0,0.000000
"Feeling down, depressed, hopeless.",0,0.000000
"Trouble falling or staying asleep, or sleeping too much.",0,0.000000
Feeling tired or having little energy,0,0.000000
Poor appetite or overeating,0,0.000000
Feeling bad about yourself or that you are a failure or have let yourself or your family down,0,0.000000
"Trouble concentrating on things, such as reading the newspaper or watching television",0,0.000000
Moving or speaking so slowly that other people could have noticed. Or the opposite being so figety or restless that you have been moving around a lot more than usual,0,0.000000


Duplicate rows: 0


In [20]:
phq9_df['type'].value_counts()

type
pre     46
post    38
Name: count, dtype: int64

In [21]:
# Check duplicate student + survey period combinations
print(
    "Duplicate student-period combinations:",
    phq9_df.duplicated(subset=["uid", "type"]).sum()
)

Duplicate student-period combinations: 0


In [22]:
# Count unique students and students with both survey periods
print("Unique students:", phq9_df["uid"].nunique())

survey_counts = pd.crosstab(phq9_df["uid"], phq9_df["type"])
print(
    "Students with both pre and post:",
    ((survey_counts["pre"] > 0) & (survey_counts["post"] > 0)).sum()
)

Unique students: 46
Students with both pre and post: 38


In [23]:
# Inspect answer categories across the nine symptom items
item_columns = phq9_df.columns[2:11]

display(
    phq9_df[item_columns]
    .melt(value_name="Answer")["Answer"]
    .value_counts(dropna=False)
)

Answer
Not at all                 412
Several days               234
More than half the days     72
Nearly every day            38
Name: count, dtype: int64

In [24]:
# Inspect the additional Response column
display(phq9_df["Response"].value_counts(dropna=False))

Response
Somewhat difficult      37
Not difficult at all    36
NaN                      5
Very difficult           3
Extremely difficult      3
Name: count, dtype: int64

**Findings and Data Cleaning Plan**

- The dataset contains 84 survey responses from 46 students: 46 pre-surveys and 38 post-surveys. We will retain the 38 students who completed both surveys for modeling because we need each student’s baseline predictors and observed end-of-term outcome to train and evaluate the model.
- We will match pre- and post-survey responses using `uid`. The 8 students without post-survey outcomes will be excluded from modeling; their missing outcomes will not be imputed.
- All nine symptom questions are complete. The additional `Response` column has 5 missing values (5.95%), but it is excluded from the PHQ-9 total, so these missing values will not cause any rows to be dropped.
- We will encode the nine symptom responses from 0 to 3 and sum them to calculate the standard PHQ-9 total (0–27).
- Pre-survey scores will serve as predictors, and post-survey PHQ-9 scores will serve as the outcome.

In [28]:
from sklearn.preprocessing import OrdinalEncoder

# Select the nine symptom questions
item_columns = phq9_df.columns[2:11]

# Define the scoring order: 0, 1, 2, 3
response_order = [
    "Not at all",
    "Several days",
    "More than half the days",
    "Nearly every day"
]

encoder = OrdinalEncoder(
    categories=[response_order for _ in item_columns],
    dtype=int
)

# Encode the nine symptom questions
phq9_clean[item_columns] = encoder.fit_transform(
    phq9_df[item_columns].apply(lambda column: column.str.strip())
)

# Calculate the total score
phq9_clean["phq9_total"] = phq9_clean[item_columns].sum(
    axis=1, min_count=9
)

phq9_clean.head()

,uid,type,Little interest or pleasure in doing things,"Feeling down, depressed, hopeless.","Trouble falling or staying asleep, or sleeping too much.",Feeling tired or having little energy,Poor appetite or overeating,Feeling bad about yourself or that you are a failure or have let yourself or your family down,"Trouble concentrating on things, such as reading the newspaper or watching television",Moving or speaking so slowly that other people could have noticed. Or the opposite being so figety or restless that you have been moving around a lot more than usual,"Thoughts that you would be better off dead, or of hurting yourself",Response,phq9_total
0,u00,pre,0,1,0,1,0,0,0,0,0,Not difficult at all,2
1,u01,pre,1,1,1,1,0,1,0,0,0,Very difficult,5
2,u02,pre,2,1,2,2,2,1,1,2,0,Somewhat difficult,13
3,u03,pre,0,1,0,0,0,0,0,1,0,Somewhat difficult,2
4,u04,pre,1,1,0,1,1,1,1,0,0,Somewhat difficult,6


In [29]:
# Create one row per student with separate pre and post scores
phq9_paired = (
    phq9_clean.pivot(
        index="uid",
        columns="type",
        values="phq9_total"
    )
    .rename(columns={
        "pre": "phq9_pre",
        "post": "phq9_post"
    })
    .rename_axis(columns=None)
    .reset_index()
)

# Exclude students without both scores
phq9_paired = phq9_paired.dropna(
    subset=["phq9_pre", "phq9_post"]
).reset_index(drop=True)

# Verify the result
print("Students retained:", len(phq9_paired))
display(phq9_paired.head())

Students retained: 38


,uid,phq9_post,phq9_pre
0,u00,3.0,2.0
1,u01,4.0,5.0
2,u02,5.0,13.0
3,u03,4.0,2.0
4,u04,8.0,6.0


2. LonelinessScale.csv — Perceived Loneliness

In [30]:
print("Shape:", loneliness_df.shape)
loneliness_df.info()

Shape: (83, 22)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83 entries, 0 to 82
Data columns (total 22 columns):
 #   Column                                                       Non-Null Count  Dtype 
---  ------                                                       --------------  ----- 
 0   uid                                                          83 non-null     object
 1   type                                                         83 non-null     object
 2   1. I feel in tune with the people around me                  83 non-null     object
 3   2. I lack companionship                                      83 non-null     object
 4   3. There is no one I can turn to                             83 non-null     object
 5   4. I do not feel alone                                       83 non-null     object
 6   5. I feel part of a group of friends                         83 non-null     object
 7   6. I have a lot in common with the people around me          83 non-null   

In [32]:
# Check survey periods
display(loneliness_df["type"].value_counts(dropna=False))

# Check duplicates
print("Duplicate rows:", loneliness_df.duplicated().sum())
print(
    "Duplicate student-period combinations:",
    loneliness_df.duplicated(subset=["uid", "type"]).sum()
)

type
pre     46
post    37
Name: count, dtype: int64

Duplicate rows: 0
Duplicate student-period combinations: 0


In [33]:
# Inspect response categories for each question
for column in loneliness_df.columns:
    if column not in ["uid", "type"]:
        print(f"\n{column}:")
        print(loneliness_df[column].value_counts(dropna=False).to_string())


1. I feel in tune with the people around me:
1. I feel in tune with the people around me
Sometimes    42
Often        31
Rarely       10

2. I lack companionship:
2. I lack companionship
Rarely       35
Sometimes    26
Never        14
Often         8

3. There is no one I can turn to:
3. There is no one I can turn to
Never        38
Rarely       24
Sometimes    16
Often         5

4. I do not feel alone:
4. I do not feel alone
Often        45
Sometimes    24
Rarely       10
Never         4

5. I feel part of a group of friends:
5. I feel part of a group of friends
Often        45
Sometimes    22
Rarely       12
Never         4

6. I have a lot in common with the people around me:
6. I have a lot in common with the people around me
Often        34
Sometimes    28
Rarely       17
Never         4

7. I am no longer close to anyone:
7. I am no longer close to anyone
Never        41
Rarely       20
Sometimes    15
Often         7

8. My interests and ideas are not shared by those around me

In [34]:
# Preserve the original data
loneliness_clean = loneliness_df.copy()

# Standardize identifiers and survey labels
loneliness_clean["uid"] = loneliness_clean["uid"].str.strip()
loneliness_clean["type"] = (
    loneliness_clean["type"].str.strip().str.lower()
)

# Keep only baseline responses
loneliness_clean = loneliness_clean.loc[
    loneliness_clean["type"] == "pre"
].copy()

# Select the 20 survey questions
loneliness_items = [
    column for column in loneliness_clean.columns
    if column not in ["uid", "type"]
]

# Inspect all baseline answer categories
display(
    loneliness_clean[loneliness_items]
    .melt(value_name="Answer")["Answer"]
    .value_counts(dropna=False)
)

print("Baseline responses retained:", len(loneliness_clean))

Answer
Sometimes    290
Often        262
Rarely       216
Never        152
Name: count, dtype: int64

Baseline responses retained: 46


**Findings and Data Cleaning Plan**

- The dataset contains 83 survey responses: 46 pre-surveys and 37 post-surveys, with 20 questionnaire items.
- No missing values, duplicate rows, or duplicate student–survey period combinations were found.
- We will retain only the 46 pre-survey responses, because baseline loneliness will be used to predict end-of-term PHQ-9 scores. Post-survey loneliness responses are not required.
- We will remove extra spaces from student IDs and response labels to ensure consistent encoding and matching.
- We will encode responses in their ordered categories and reverse-score the appropriate items so that higher scores consistently indicate greater loneliness.
- We will calculate the total loneliness score using all 20 correctly scored items and check for missing scores and values outside the expected range.
- We will retain `uid` and `loneliness_pre` for merging with the PHQ-9 table. Only students with the required observed PHQ-9 outcome will enter supervised modeling.

In [36]:
# Create a copy and standardize IDs and survey labels
loneliness_clean = loneliness_df.copy()

loneliness_clean["uid"] = loneliness_clean["uid"].str.strip()
loneliness_clean["type"] = (
    loneliness_clean["type"].str.strip().str.lower()
)

In [37]:
# keep only pre survey response
loneliness_clean = loneliness_clean.loc[
    loneliness_clean["type"] == "pre"
].copy()

print("Baseline students retained:", len(loneliness_clean))

Baseline students retained: 46


In [38]:
# Select the 20 questions and clean response labels
loneliness_items = [
    column for column in loneliness_clean.columns
    if column not in ["uid", "type"]
]

loneliness_clean[loneliness_items] = (
    loneliness_clean[loneliness_items]
    .apply(lambda column: column.str.strip())
)

In [40]:
# Check for missing responses and duplicate students
assert len(loneliness_items) == 20, "Expected 20 questions."

assert loneliness_clean["uid"].notna().all(), \
    "Missing student IDs."

assert not loneliness_clean["uid"].duplicated().any(), \
    "Duplicate baseline responses."

assert loneliness_clean[loneliness_items].notna().all().all(), \
    "Missing questionnaire responses."

In [41]:
# Encode responses from 1 to 4
from sklearn.preprocessing import OrdinalEncoder

response_order = ["Never", "Rarely", "Sometimes", "Often"]

loneliness_encoder = OrdinalEncoder(
    categories=[response_order for _ in loneliness_items],
    dtype=int
)

loneliness_clean[loneliness_items] = (
    loneliness_encoder.fit_transform(
        loneliness_clean[loneliness_items]
    ) + 1
)

In [42]:
# Reverse-score the positively worded questions
reverse_numbers = {1, 4, 5, 6, 9, 10, 15, 16, 19, 20}

reverse_columns = [
    column for column in loneliness_items
    if int(column.split(".")[0].strip()) in reverse_numbers
]

assert len(reverse_columns) == 10, \
    "Check reverse-scored column names."

loneliness_clean[reverse_columns] = (
    5 - loneliness_clean[reverse_columns]
)

In [43]:
loneliness_clean.head()

,uid,type,1. I feel in tune with the people around me,2. I lack companionship,3. There is no one I can turn to,4. I do not feel alone,5. I feel part of a group of friends,6. I have a lot in common with the people around me,7. I am no longer close to anyone,8. My interests and ideas are not shared by those around me,...,11. I feel left out,12. My social relationships are superficial,13. No one really knows me well,14. I feel isolated from others,15. I can find companionship when I want it,16. There are people who really understand me,17. I am unhappy being so withdrawn,18. People are around me but not with me,19. There are people I can talk to,20. There are people I can turn to
0,u00,pre,2,2,1,4,1,2,1,1,...,2,2,1,1,1,3,4,3,1,1
1,u01,pre,2,2,2,2,1,1,1,1,...,2,2,1,2,1,1,3,1,1,1
2,u02,pre,2,2,2,1,1,1,1,4,...,2,1,1,1,1,1,1,1,1,1
3,u03,pre,2,2,1,1,3,3,3,3,...,2,2,3,3,2,1,3,3,2,2
4,u04,pre,2,3,3,2,3,2,1,2,...,3,3,3,3,3,3,3,3,3,3


In [44]:
# calculate and validate total score
loneliness_clean["loneliness_pre"] = (
    loneliness_clean[loneliness_items].sum(axis=1, min_count=20)
)

assert loneliness_clean["loneliness_pre"].between(20, 80).all(), \
    "Missing or out-of-range totals found."

In [46]:
# prepare the table for merging
loneliness_pre = loneliness_clean[
    ["uid", "loneliness_pre"]
].reset_index(drop=True)

display(loneliness_pre.head())

,uid,loneliness_pre
0,u00,35
1,u01,30
2,u02,29
3,u03,47
4,u04,53


3. FlourishingScale.csv — Psychological Well-Being

In [47]:
print("Shape:", flourishing_df.shape)
flourishing_df.info()

Shape: (83, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83 entries, 0 to 82
Data columns (total 10 columns):
 #   Column                                                                 Non-Null Count  Dtype  
---  ------                                                                 --------------  -----  
 0   uid                                                                    83 non-null     object 
 1   type                                                                   83 non-null     object 
 2   I lead a purposeful and meaningful life                                83 non-null     int64  
 3   My social relationships are supportive and rewarding                   83 non-null     int64  
 4   I am engaged and interested in my daily activities                     80 non-null     float64
 5   I actively contribute to the happiness and well-being of others        82 non-null     float64
 6   I am competent and capable in the activities that are important to me  82 no

In [48]:
display(flourishing_df["type"].value_counts(dropna=False))

print("Duplicate rows:", flourishing_df.duplicated().sum())

print(
    "Duplicate student-period combinations:",
    flourishing_df.duplicated(subset=["uid", "type"]).sum()
)

type
pre     46
post    37
Name: count, dtype: int64

Duplicate rows: 0
Duplicate student-period combinations: 0


In [49]:
for column in flourishing_df.columns:
    if column not in ["uid", "type"]:
        print(f"\n{column}:")
        print(
            flourishing_df[column]
            .value_counts(dropna=False)
            .to_string()
        )


I lead a purposeful and meaningful life:
I lead a purposeful and meaningful life
6    29
7    23
5    18
3     6
4     4
2     2
1     1

My social relationships are supportive and rewarding:
My social relationships are supportive and rewarding
6    32
5    18
7    16
2     6
4     6
3     3
1     2

I am engaged and interested in my daily activities:
I am engaged and interested in my daily activities
6.0    24
5.0    19
4.0    15
7.0    11
2.0     7
3.0     4
NaN     3

I actively contribute to the happiness and well-being of others:
I actively contribute to the happiness and well-being of others
6.0    34
5.0    22
4.0     8
3.0     8
7.0     5
1.0     3
2.0     2
NaN     1

I am competent and capable in the activities that are important to me:
I am competent and capable in the activities that are important to me
6.0    29
7.0    18
5.0    18
4.0    10
2.0     4
3.0     3
NaN     1

I am a good person and live a good life:
I am a good person and live a good life
7.0    32
6.0    27


The flourishing survey has 83 responses: 46 pre and 37 post, with no duplicates. Its eight questionnaire items are already numeric, so no encoder is needed.

There are 6 missing responses across four questions, but we first need to check whether they occur in the pre-surveys we will use.

In [64]:
# Create a copy and standardize IDs and labels
flourishing_clean = flourishing_df.copy()

flourishing_clean["uid"] = flourishing_clean["uid"].str.strip()
flourishing_clean["type"] = (
    flourishing_clean["type"].str.strip().str.lower()
)

In [65]:
# Keep only pre-survey responses
flourishing_clean = flourishing_clean.loc[
    flourishing_clean["type"] == "pre"
].copy()

print("Baseline students retained:", len(flourishing_clean))

Baseline students retained: 46


In [66]:
# Select the eight questions and check baseline missingness
flourishing_items = [
    column for column in flourishing_clean.columns
    if column not in ["uid", "type"]
]

display(flourishing_clean[flourishing_items].isna().sum())

print(
    "Students with at least one missing item:",
    flourishing_clean[flourishing_items].isna().any(axis=1).sum()
)

I lead a purposeful and meaningful life                                  0
My social relationships are supportive and rewarding                     0
I am engaged and interested in my daily activities                       3
I actively contribute to the happiness and well-being of others          1
I am competent and capable in the activities that are important to me    1
I am a good person and live a good life                                  1
I am optimistic about my future                                          0
People respect me                                                        0
dtype: int64

Students with at least one missing item: 4


In [67]:
# Inspect the numeric response values
display(
    flourishing_clean[flourishing_items]
    .melt(value_name="Score")["Score"]
    .value_counts(dropna=False)
    .sort_index()
)

Score
1.0      6
2.0     17
3.0     17
4.0     42
5.0     76
6.0    129
7.0     75
NaN      6
Name: count, dtype: int64

The baseline flourishing data shows:
- 6 missing item responses across 4 students.
- 42 students have complete responses.
- All observed item values are between 1 and 7.
For now, we are keeping all 46 students and leaving incomplete totals missing. After merging, we can see how many of those four students have PHQ-9 outcomes before deciding how to handle missing predictors.

In [68]:
# Validate the observed item values
assert len(flourishing_items) == 8, "Expected eight questions."

assert (
    flourishing_clean[flourishing_items].isin(range(1, 8))
    | flourishing_clean[flourishing_items].isna()
).all().all(), "Unexpected item values found."

In [69]:
# Calculate totals only for complete surveys
flourishing_clean["flourishing_pre"] = (
    flourishing_clean[flourishing_items].sum(axis=1, min_count=8)
)

In [70]:
# Prepare the table for merging
flourishing_pre = flourishing_clean[
    ["uid", "flourishing_pre"]
].reset_index(drop=True)

print("Students retained:", len(flourishing_pre))
print("Missing totals:", flourishing_pre["flourishing_pre"].isna().sum())

display(flourishing_pre.head())

Students retained: 46
Missing totals: 4


,uid,flourishing_pre
0,u00,NaN
1,u01,45.0
2,u02,46.0
3,u03,34.0
4,u04,27.0


### Dataset for the Model

In [71]:
# Add baseline loneliness to the paired PHQ-9 table
student_data = phq9_paired.merge(
    loneliness_pre,
    on="uid",
    how="left",
    validate="one_to_one"
)
# Add baseline flourishing
student_data = student_data.merge(
    flourishing_pre,
    on="uid",
    how="left",
    validate="one_to_one"
)
student_data.head()

,uid,phq9_post,phq9_pre,loneliness_pre,flourishing_pre
0,u00,3.0,2.0,35,NaN
1,u01,4.0,5.0,30,45.0
2,u02,5.0,13.0,29,46.0
3,u03,4.0,2.0,47,34.0
4,u04,8.0,6.0,53,27.0


In [72]:
print("Shape:", student_data.shape)
print("Unique students:", student_data["uid"].nunique())

display(student_data.isna().sum())

assert student_data["uid"].is_unique
assert student_data["phq9_post"].notna().all()

Shape: (38, 5)
Unique students: 38


uid                0
phq9_post          0
phq9_pre           0
loneliness_pre     0
flourishing_pre    3
dtype: int64

### Test-Train Split

In [73]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    student_data,
    test_size=0.20,
    random_state=42
)

print("Training students:", len(train_data))
print("Test students:", len(test_data))

Training students: 30
Test students: 8


### Explanatory Data Analysis on Training Data

In [75]:
student_train = train_data.copy()

In [76]:
student_train.describe()

,phq9_post,phq9_pre,loneliness_pre,flourishing_pre
count,30.000000,30.000000,30.000000,28.000000
mean,6.066667,5.433333,37.833333,43.142857
std,6.073875,4.952765,11.546259,8.154591
min,0.000000,0.000000,23.000000,16.000000
25%,2.000000,2.000000,30.000000,40.500000
50%,4.000000,4.500000,35.000000,45.500000
75%,7.000000,7.000000,46.000000,48.000000
max,25.000000,23.000000,65.000000,54.000000
